In [0]:
control_table = "dbw_healthcare_de_dev.etl.file_control"

df_files = spark.sql(f"""
    SELECT
        file_id,
        batch_id,
        source_container,
        source_folder,
        file_name,
        load_type,
        processing_stage,
        pipeline_status,
        retry_count
    FROM {control_table}
    WHERE is_enabled = TRUE
      AND (
          (pipeline_status = 'PENDING'
           AND processing_stage = 'REGISTERED')
          OR
          (pipeline_status = 'FAILED'
           AND retry_count < 3)
      )
    ORDER BY registered_at, file_id
""")

display(df_files)

In [0]:
import json

# Convert eligible file records into dictionaries
files = [row.asDict() for row in df_files.collect()]

# Build the JSON response
result = {
    "file_count": len(files),
    "files": files
}

# Convert Python dictionary to JSON
result_json = json.dumps(result)

print(result_json)
dbutils.notebook.exit(result_json)